# 04 — Analisis sensitivitas: gula darah puasa (subsampel puasa)

**Tujuan:** menjawab pertanyaan reviewer, *apakah kesenjangan pada NH Asian hanya artefak HbA1c?*

Di subsampel puasa NHANES (peserta yang diperiksa pagi hari setelah puasa ≥ 8 jam), model regresi logistik yang sama
dijalankan ulang untuk tiga definisi outcome:

| Kode | Definisi | Peran |
| --- | --- | --- |
| `A1C` | HbA1c ≥ 6,5% | pembanding (definisi naskah, di subsampel yang sama) |
| `A1C_ATAU_GDP` | HbA1c ≥ 6,5% **atau** gula darah puasa ≥ 126 mg/dL | **analisis sensitivitas utama** (definisi ADA) |
| `GDP` | gula darah puasa ≥ 126 mg/dL saja | definisi tanpa HbA1c |

Bobot yang dipakai adalah **bobot subsampel puasa** (WTSAF2YR / WTSAFPRP), digabung antarsiklus dengan faktor yang sama
seperti bobot MEC (2/9,2 dan 3,2/9,2). Prosedur model, rekalibrasi, threshold, dan bootstrap sama dengan notebook 02.

**Cara menjalankan:** taruh notebook ini di folder yang sama dengan notebook 02, lalu jalankan dari atas ke bawah.
Butuh internet untuk mengunduh 4 file GLU dari CDC. Mode penuh (`CEPAT = False`) sekitar 10–20 menit.

In [ ]:
from pathlib import Path
import time, warnings, urllib.request
import numpy as np
import pandas as pd
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_predict
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

# folder root proyek: berisi folder data/ dan hasil/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw"
DATA = ROOT / "data" / "processed" / "nhanes_dm_belum_terdiagnosis.csv"
HASIL = ROOT / "hasil"
RAW.mkdir(parents=True, exist_ok=True)

CEPAT = False              # True = uji kilat (tuning dan bootstrap dikurangi)
RANDOM_STATE = 42
N_LIPATAN = 5
N_ITER_TUNING = 4 if CEPAT else 15
N_BOOT = 100 if CEPAT else 500
TARGET_SENSITIVITAS = 0.80
MIN_KASUS = 30
GDP_CUTOFF = 126           # mg/dL
print("Folder kerja:", ROOT, "| mode cepat:", CEPAT)

## 1. Unduh file gula darah puasa (GLU) dari CDC
File yang sudah ada dilewati. Kalau gagal, unduh manual dari situs NHANES dan simpan di `data\raw` dengan nama yang sama.

In [ ]:
BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}.xpt"
GLU = {  # siklus -> (tahun URL, nama file, lama siklus)
    "2011-2012": ("2011", "GLU_G", 2.0),
    "2013-2014": ("2013", "GLU_H", 2.0),
    "2015-2016": ("2015", "GLU_I", 2.0),
    "2017-2020": ("2017", "P_GLU", 3.2),
}
TOTAL_TAHUN = 9.2


def unduh(url, tujuan, percobaan=3):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (research download)"})
    for i in range(1, percobaan + 1):
        try:
            with urllib.request.urlopen(req, timeout=120) as resp:
                isi = resp.read()
            if not isi.startswith(b"HEADER RECORD"):
                raise ValueError("isi bukan file XPT")
            tujuan.write_bytes(isi)
            return True
        except Exception as e:
            print(f"   percobaan {i} gagal: {e}")
            time.sleep(3 * i)
    return False


for siklus, (tahun, nama, _) in GLU.items():
    tujuan = RAW / f"{nama}.xpt"
    if tujuan.exists() and tujuan.stat().st_size > 0:
        print(f"[ada]    {nama}")
    elif unduh(BASE_URL.format(year=tahun, name=nama), tujuan):
        print(f"[unduh]  {nama} selesai")
    else:
        print(f"[GAGAL]  {nama} — unduh manual")

## 2. Gabungkan dengan dataset analisis dan bentuk subsampel puasa

In [ ]:
def baca_xpt(nama):
    return pd.read_sas(RAW / f"{nama}.xpt", format="xport", encoding="utf-8")


bagian = []
for siklus, (_, nama, lama) in GLU.items():
    g = baca_xpt(nama)
    kol_bobot = [c for c in g.columns if c.upper().startswith("WTSAF")][0]
    bagian.append(pd.DataFrame({"SEQN": g["SEQN"].astype("int64"), "GDP": g["LBXGLU"],
                                "WT_PUASA": g[kol_bobot] * lama / TOTAL_TAHUN}))
    print(f"{siklus}: {len(g):,} baris, kolom bobot {kol_bobot}")
glu = pd.concat(bagian, ignore_index=True)

d0 = pd.read_csv(DATA)
d0.loc[d0["DBP"] < 30, "DBP"] = np.nan
d0.loc[d0["SBP"] < 60, "SBP"] = np.nan
d0["PEREMPUAN"] = (d0["JENIS_KELAMIN"] == "Perempuan").astype(int)
d = d0.merge(glu, on="SEQN", how="left")
alur = [("Sampel analisis naskah", len(d))]
d = d[d["WT_PUASA"].notna() & (d["WT_PUASA"] > 0)]
alur.append(("Masuk subsampel puasa (bobot puasa > 0)", len(d)))
d = d[d["GDP"].notna()].reset_index(drop=True)
alur.append(("Punya nilai gula darah puasa", len(d)))

d["Y_A1C"] = (d["HBA1C"] >= 6.5).astype(int)
d["Y_GDP"] = (d["GDP"] >= GDP_CUTOFF).astype(int)
d["Y_A1C_ATAU_GDP"] = ((d["Y_A1C"] == 1) | (d["Y_GDP"] == 1)).astype(int)
OUTCOME = {"A1C": "Y_A1C", "A1C_ATAU_GDP": "Y_A1C_ATAU_GDP", "GDP": "Y_GDP"}

PREDIKTOR = ["USIA", "PEREMPUAN", "BMI", "LINGKAR_PINGGANG", "SBP", "DBP", "RIWAYAT_HIPERTENSI",
             "RIWAYAT_KELUARGA_DM", "PERNAH_MEROKOK", "AKTIF_FISIK", "PENDIDIKAN"]
ATRIBUT = {"RAS_ETNIS": "Ras/etnis", "KEL_USIA": "Usia"}
X = d[PREDIKTOR].astype(float).values
W = d["WT_PUASA"].values

print()
print(pd.DataFrame(alur, columns=["Tahap", "N"]).to_string(index=False))
print()
for k, kol in OUTCOME.items():
    print(f"{k:14s} kasus {int(d[kol].sum()):5,}  prevalensi berbobot {100 * np.average(d[kol], weights=W):.2f}%")
# kasus per ras/etnis untuk tiap definisi
pd.DataFrame({k: d.groupby("RAS_ETNIS")[kol].sum() for k, kol in OUTCOME.items()})

## 3. Nested cross-validation regresi logistik (sama dengan notebook 02)

In [ ]:
def logit(q):
    q = np.clip(q, 1e-6, 1 - 1e-6)
    return np.log(q / (1 - q))


def pilih_threshold(y, p, w, target=TARGET_SENSITIVITAS):
    pos = y == 1
    urut = np.argsort(-p[pos])
    p_pos, w_pos = p[pos][urut], w[pos][urut]
    kum = np.cumsum(w_pos) / w_pos.sum()
    return p_pos[np.searchsorted(kum, target)]


MODEL_LR = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                     ("clf", LogisticRegression(class_weight="balanced", max_iter=5000))])
RUANG_LR = {"clf__C": loguniform(1e-3, 10)}


def nested_cv(y):
    strat = d["SIKLUS"].astype(str) + "_" + y.astype(str)
    luar = StratifiedKFold(n_splits=N_LIPATAN, shuffle=True, random_state=RANDOM_STATE)
    prob = np.zeros(len(y)); pred = np.zeros(len(y), dtype=int)
    for tr, te in luar.split(X, strat):
        cari = RandomizedSearchCV(clone(MODEL_LR), RUANG_LR, n_iter=N_ITER_TUNING, scoring="average_precision",
                                  cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                                  random_state=RANDOM_STATE, n_jobs=1)
        cari.fit(X[tr], y[tr])
        terbaik = cari.best_estimator_
        oof = cross_val_predict(clone(terbaik), X[tr], y[tr], method="predict_proba",
                                cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE))[:, 1]
        platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(oof).reshape(-1, 1), y[tr],
                                                             sample_weight=W[tr] / W[tr].mean())
        kal = lambda q: platt.predict_proba(logit(q).reshape(-1, 1))[:, 1]
        thr = pilih_threshold(y[tr], kal(oof), W[tr])
        prob[te] = kal(terbaik.predict_proba(X[te])[:, 1])
        pred[te] = (prob[te] >= thr).astype(int)
    return prob, pred


PRED = {}
for k, kol in OUTCOME.items():
    mulai = time.time()
    y = d[kol].values.astype(int)
    PRED[k] = (y,) + nested_cv(y)
    print(f"{k:14s} selesai ({(time.time() - mulai) / 60:.1f} menit)")

## 4. Metrik, kalibrasi per kelompok, dan survey bootstrap
Sama dengan notebook 02, ditambah *calibration intercept* (slope ditetapkan 1) dan *calibration slope* per kelompok.

In [ ]:
def bobot_replikasi(n_boot, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    psu = d["SDMVSTRA"].astype(int).astype(str) + "_" + d["SDMVPSU"].astype(int).astype(str)
    faktor = np.ones((len(d), n_boot), dtype=np.float32)
    for strata, g in d.groupby("SDMVSTRA"):
        daftar = psu[g.index].unique()
        n_h = len(daftar)
        if n_h < 2:
            continue
        for b in range(n_boot):
            ambil = rng.choice(daftar, size=n_h - 1, replace=True)
            hitung = pd.Series(ambil).value_counts()
            faktor[g.index, b] = psu[g.index].map(hitung).fillna(0).values * n_h / (n_h - 1)
    return faktor


def kalibrasi(y, p, w):
    lp = logit(p)
    slope = LogisticRegression(C=1e6, max_iter=1000).fit(lp.reshape(-1, 1), y, sample_weight=w / w.mean()).coef_[0][0]
    a = 0.0
    for _ in range(50):
        q = 1 / (1 + np.exp(-(lp + a)))
        a += np.sum(w * (y - q)) / np.sum(w * q * (1 - q))
    return a, slope


def metrik(y, p, yhat, w):
    pos, neg = y == 1, y == 0
    if w[pos].sum() == 0 or w[neg].sum() == 0:
        return dict.fromkeys(["AUROC", "Sensitivitas", "Spesifisitas", "PPV", "FNR", "FPR", "O/E", "Intercept", "Slope"], np.nan)
    tp = w[pos & (yhat == 1)].sum(); fn = w[pos & (yhat == 0)].sum()
    tn = w[neg & (yhat == 0)].sum(); fp = w[neg & (yhat == 1)].sum()
    a, s = kalibrasi(y, p, w)
    return {"AUROC": roc_auc_score(y, p, sample_weight=w), "Sensitivitas": tp / (tp + fn), "Spesifisitas": tn / (tn + fp),
            "PPV": tp / (tp + fp) if tp + fp > 0 else np.nan, "FNR": fn / (tp + fn), "FPR": fp / (tn + fp),
            "O/E": np.average(y, weights=w) / np.average(p, weights=w), "Intercept": a, "Slope": s}


def dengan_ci(titik, rep):
    out = {}
    for k, v in titik.items():
        out[k] = v
        out[k + "_lo"] = np.nanpercentile(rep[k], 2.5)
        out[k + "_hi"] = np.nanpercentile(rep[k], 97.5)
    return out


FAKTOR = bobot_replikasi(N_BOOT)
ringkasan, subkelompok, kesenjangan = [], [], []
for k in OUTCOME:
    mulai = time.time()
    y, p, yhat = PRED[k]
    kelompok = [("Keseluruhan", "Semua", np.ones(len(d), bool))]
    for atr, label in ATRIBUT.items():
        for g in sorted(d[atr].dropna().astype(str).unique()):
            kelompok.append((label, g, (d[atr].astype(str) == g).values))
    rep = {}
    for label, g, m in kelompok:
        titik = metrik(y[m], p[m], yhat[m], W[m])
        r = pd.DataFrame([metrik(y[m], p[m], yhat[m], W[m] * FAKTOR[m, b]) for b in range(N_BOOT)])
        rep[(label, g)] = r
        baris = {"Outcome": k, "Atribut": label, "Kelompok": g, "N": int(m.sum()), "Kasus": int(y[m].sum()),
                 "Cukup_kasus": int(y[m].sum()) >= MIN_KASUS, **dengan_ci(titik, r)}
        (ringkasan if label == "Keseluruhan" else subkelompok).append(baris)
    for label in ATRIBUT.values():
        cukup = [g for (l, g, m) in kelompok if l == label and y[m].sum() >= MIN_KASUS]
        if len(cukup) < 2:
            continue
        fnr_titik = {g: [b for b in subkelompok if b["Outcome"] == k and b["Atribut"] == label and b["Kelompok"] == g][0]["FNR"]
                     for g in cukup}
        fnr_rep = pd.concat({g: rep[(label, g)]["FNR"] for g in cukup}, axis=1)
        sel_rep = fnr_rep.max(axis=1) - fnr_rep.min(axis=1)
        kesenjangan.append({"Outcome": k, "Atribut": label, "Kelompok_dibandingkan": len(cukup),
                            "FNR_tertinggi": max(fnr_titik, key=fnr_titik.get), "FNR_terendah": min(fnr_titik, key=fnr_titik.get),
                            "Selisih_FNR": max(fnr_titik.values()) - min(fnr_titik.values()),
                            "Selisih_FNR_lo": np.nanpercentile(sel_rep, 2.5), "Selisih_FNR_hi": np.nanpercentile(sel_rep, 97.5)})
    print(f"{k:14s} bootstrap selesai ({(time.time() - mulai) / 60:.1f} menit)")

ringkasan = pd.DataFrame(ringkasan); subkelompok = pd.DataFrame(subkelompok); kesenjangan = pd.DataFrame(kesenjangan)
ringkasan.to_csv(HASIL / "05_sensitivitas_puasa_ringkasan.csv", index=False)
subkelompok.to_csv(HASIL / "05_sensitivitas_puasa_subkelompok.csv", index=False)
kesenjangan.to_csv(HASIL / "05_sensitivitas_puasa_kesenjangan.csv", index=False)
pd.DataFrame(alur, columns=["Tahap", "N"]).to_csv(HASIL / "05_sensitivitas_puasa_alur.csv", index=False)
print("Hasil disimpan di folder hasil (file 05_sensitivitas_puasa_*.csv).")

## 5. Ringkasan untuk dibaca
Yang penting dilihat: apakah NH Asian tetap punya FNR tertinggi dan *calibration intercept* positif pada definisi `A1C_ATAU_GDP` dan `GDP`.

In [ ]:
print("Performa keseluruhan")
print(ringkasan[["Outcome", "N", "Kasus", "AUROC", "Sensitivitas", "Spesifisitas", "O/E", "Intercept", "Slope"]].round(3).to_string(index=False))
print()
print("Kesenjangan FNR (kelompok dengan >= 30 kasus)")
print(kesenjangan.round(3).to_string(index=False))
print()
ras = subkelompok[subkelompok.Atribut == "Ras/etnis"]
print("Per ras/etnis: FNR dan calibration intercept")
ras.pivot_table(index="Kelompok", columns="Outcome", values=["Kasus", "FNR", "Intercept"]).round(2)

## Hasil yang tersimpan
| File | Isi |
| --- | --- |
| `hasil\05_sensitivitas_puasa_ringkasan.csv` | Performa keseluruhan tiga definisi outcome, dengan CI 95% |
| `hasil\05_sensitivitas_puasa_subkelompok.csv` | FNR, FPR, AUROC, O/E, intercept, slope per ras/etnis dan usia |
| `hasil\05_sensitivitas_puasa_kesenjangan.csv` | Selisih FNR per atribut, dengan CI 95% |
| `hasil\05_sensitivitas_puasa_alur.csv` | Alur subsampel puasa |

Setelah selesai, kirim ke Claude folder `hasil` (atau keempat file di atas) supaya hasilnya ditulis ke naskah.